# Домашнє завдання: Прогнозування орендної плати за житло

## Мета завдання
Застосувати знання з лекції для побудови моделі лінійної регресії, що прогнозує орендну плату за житло в Індії. Ви пройдете весь цикл вирішення задачі машинного навчання: від дослідницького аналізу до оцінки якості моделі.

## Опис датасету
**House Rent Prediction Dataset** містить інформацію про 4700+ оголошень про оренду житла в Індії з такими параметрами:
- **BHK**: Кількість спалень, залів, кухонь
- **Rent**: Орендна плата (цільова змінна)
- **Size**: Площа в квадратних футах
- **Floor**: Поверх та загальна кількість поверхів
- **Area Type**: Тип розрахунку площі
- **Area Locality**: Район
- **City**: Місто
- **Furnishing Status**: Стан меблювання
- **Tenant Preferred**: Тип орендаря
- **Bathroom**: Кількість ванних кімнат
- **Point of Contact**: Контактна особа

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `House_Rent_Dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (включно з типами даних та кількістю значень)


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px


In [3]:
df = pd.read_csv('House_Rent_Dataset.csv') # read data

In [4]:
df.shape # size of df

(4746, 12)

In [5]:
df.head() # first 5 rows

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [6]:
df.info() # main info


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


## Завдання 2: Дослідницький аналіз даних (EDA) (5 балів)

**Що потрібно зробити:**
1. **Аналіз пропущених значень.** Перевірте наявність і відсоток пропущених значень у кожній колонці
2. **Базова статистика.** Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.
3. **Аналіз цільової змінної.** Побудуйте гістограму розподілу цільової змінної (Rent)
4. **Робота з викидами.** Знайдіть та видаліть викиди в цільовій змінній (якщо є). Визначити викиди можна будь-яким зрозумілим для вас способом, як варіант - таким, що використовується в побудові box-plot (https://en.wikipedia.org/wiki/Box_plot#Example_with_outliers).
5. **Аналіз категоріальних змінних.** Виведіть кількість унікальних значень для кожної з категоріальних колонок.


In [7]:
# Аналіз пропущених значень

# із результату df.info() видно, що в кожній колонці кількість non-null значень співпадає із загальною кількістю записів
# отже null значень в базі немає

# додатково це перевіримо
missing_data = pd.DataFrame({
    'count': df.isnull().sum(),
    'percent': (df.isnull().mean() * 100).round(2)})
missing_data

,count,percent
Posted On,0,0.0
BHK,0,0.0
Rent,0,0.0
Size,0,0.0
Floor,0,0.0
Area Type,0,0.0
Area Locality,0,0.0
City,0,0.0
Furnishing Status,0,0.0
Tenant Preferred,0,0.0


In [8]:
stats = df.describe() # Базова статистика
stats.round(2)

,BHK,Rent,Size,Bathroom
count,4746.00,4746.00,4746.00,4746.00
mean,2.08,34993.45,967.49,1.97
std,0.83,78106.41,634.20,0.88
min,1.00,1200.00,10.00,1.00
25%,2.00,10000.00,550.00,1.00
50%,2.00,16000.00,850.00,2.00
75%,3.00,33000.00,1200.00,2.00
max,6.00,3500000.00,8000.00,10.00


Середнє-арифметичне вартості оренди (35 тис) суттєво вище за медіану (16 тис), та максимум 3.5 млн., що вказує на суттєву скошеність розподілу вправо.

Також, мінімум площі становить 10 кв футів, що виглядає як помилка.

In [9]:
# Аналіз цільової змінної
# Розподіл оренди
fig = px.histogram(
    df,
    x='Rent',
    nbins=200,
    title='Розподіл цільової змінної (вартість оренди)',
    labels={'Rent': 'Вартість оренди'}
)
fig.update_layout(
    showlegend=False,
    height=400,
    yaxis_title='Кількість оголошень'
)
fig.show()


Гістограма підтверджує висновок щодо сильної скошеності розподілу цільової змінної вправо.

In [10]:
# Робота з викидами Rent
px.box(df, y='Rent').show() #boxplot with outliers

Графік має такий вигляд, коли коробку розподілу не видно, з причини сильної правоскошеності, на яку було вказано вище.

вартість оренди 75% даних не перевищує 33 000 (це видно із базової статистики, і у впливаючому вікні на графіку), а у вищих перцентилях зростає до 3.5 млн.

Тому використаємо правило інтерквартильного діапазону (IQR rule), щоби визначити викиди і очистити дані від них.


In [11]:
# Використаємо правило інтерквартального діапазону (IQR)
iqr = df['Rent'].quantile(0.75) - df['Rent'].quantile(0.25)
lower_fence = df['Rent'].quantile(0.25) - 1.5*iqr
upper_fence = df['Rent'].quantile(0.75) + 1.5*iqr
print(f'Викиди вартості оренди: \n ціна оренди менше: {lower_fence} \n ціна оренди більше: {upper_fence}')


Викиди вартості оренди: 
 ціна оренди менше: -24500.0 
 ціна оренди більше: 67500.0


Статистичних викидів зліва фактично немає, тому видаляємо викиди справа, а квартири з низькою вартістю оренди проаналізуємо додатково на наявність помилок в даних.

In [12]:
# видалення викидів

df_clean = df[df['Rent']<= upper_fence]
print(f'Видалено {len(df) - len(df_clean)} рядків ({(1 - len(df_clean)/len(df)):.1%})')


Видалено 520 рядків (11.0%)


In [13]:
# перевірка аномально низьких значень
df.nsmallest(5, 'Rent')

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
4076,2022-05-31,3,1200,2100,1 out of 3,Carpet Area,"Uppal, NH 2 2",Hyderabad,Furnished,Bachelors/Family,3,Contact Owner
285,2022-05-24,1,1500,200,Ground out of 2,Super Area,Santoshpur,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
471,2022-05-12,1,1800,500,Ground out of 1,Super Area,Shyam Bazar,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2475,2022-06-22,2,2000,60,1 out of 1,Super Area,Ram Nagar,Delhi,Unfurnished,Bachelors/Family,1,Contact Owner
146,2022-06-05,2,2200,550,2 out of 3,Super Area,Behala silpara,Kolkata,Furnished,Bachelors/Family,1,Contact Owner


Рядок з індексом 4076 ймовірно має помилку у вартості оренди (квартира має 3 приміщення, 3 ванні кімнати, велику площу, а ціна оренди низька). Можливо одрук або ціна вказана не у рупіях, а іншій (більш дорогій валюті).

Рядок з індексом 2475 має нереалістичну площу - 60 кв футів (~5.6 кв м) і 2 приміщення.

Перевіримо наявність аномально маленьких квартир (менше 150 кв футів або 14 кв м):

In [14]:
# аналіз квартир з площею до 150 кв ф
small_flats = df[df['Size'] < 150]
small_flats.groupby(['City', 'BHK']).agg(flats_cnt=('Size', 'count'), av_rent=('Rent', 'mean'))

flats_cnt       av_rent
City      BHK                         
Bangalore 1            4   5725.000000
          2            2  12000.000000
Chennai   1            1   4500.000000
          2            1  15000.000000
          3            2  11500.000000
Delhi     1           27   9277.777778
          2           65  15656.923077
          3           16  19468.750000
          4            1  15000.000000
Hyderabad 1           11   5663.636364
          2           19  16789.473684
          3            3  18333.333333
          4            1  10000.000000
Kolkata   1            4   5250.000000
          3            1  34000.000000

Квартири розміром менше 150 кв.футів виглядають нереалістично, особливо в тих випадках, коли приміщень заявлено більше, ніж 1 і ціна оренди висока (медіанна, або більше).

Тому скоріше всього - це помилки в даних (наприклад, площа квартири вказана не у кв.футах, а у кв.ярдах)


In [15]:
# Видалення аномальних рядків:
# Size < 150 & індекс 4076
before = len(df_clean)
df_clean = df_clean[(df_clean['Size'] >= 150) & (df_clean.index != 4076)]
print(f'Видалено аномальних рядків: {before - len(df_clean)} ({(before - len(df_clean)) / before:.1%})')


Видалено аномальних рядків: 158 (3.7%)


Окрім викидів у Rent, у даних є аномалії, які не є статистичними викидами, а схожі на помилки введення: площа 10–150 кв. футів для квартир із 2–4 спальнями.

Також рядок з індексом 4076: оренда 1200 за 2100 кв. футів — ймовірно,одрук.

Ці рядки видалено (~3.7% даних), бо вони спотворювали б зв'язок між площею та ціною.

In [16]:
 # boxplot без викидів
px.box(df_clean, y='Rent').show()

Тепер boxplot показує більш рівномірний розподіл: він теж скошений вправо, але вже не так сильно.

Глянемо на базову статистику:

In [17]:
# Базова статистика на очищених даних без викидів
stats_2 = df_clean.describe()
stats_2.round(2)

,BHK,Rent,Size,Bathroom
count,4068.00,4068.00,4068.00,4068.00
mean,1.96,19529.68,901.89,1.81
std,0.75,13955.20,468.90,0.71
min,1.00,1500.00,150.00,1.00
25%,1.00,9500.00,550.00,1.00
50%,2.00,15000.00,800.00,2.00
75%,2.00,25000.00,1140.00,2.00
max,6.00,67000.00,4200.00,7.00


Середнє-арифметичне вартості оренди зменшилось з 35 тис до 19.5 тис, і стало суттєво ближче до медіани (15 тис). Максимум становить 67 тис, що все ще вказує на скошеність розподілу вправо, але вже значно менше.

Також, мінімум площі в очищеному наборі даних становить 150 кв футів.

In [18]:
# Аналіз категоріальних змінних. Виведіть кількість унікальних значень для кожної з категоріальних колонок.
df_clean.select_dtypes('object').nunique()

,0
Posted On,80
Floor,340
Area Type,3
Area Locality,1947
City,6
Furnishing Status,3
Tenant Preferred,3
Point of Contact,3


Area Locality (1947 значень), Floor (340) і Posted On (80) мають забагато унікальних значень для One-Hot Encoding.

City, Area Type, Furnishing Status, Tenant Preferred і Point of Contact мають 3–6 значень, тож їх можна кодувати через One-Hot Encoding.



## Завдання 3: Аналіз кореляцій та взаємозв'язків (3 бали)

**Що потрібно зробити:**
1. Обчисліть матрицю кореляцій для числових змінних
2. Візуалізуйте кореляційну матрицю за допомогою heatmap
3. Побудуйте scatter plot між Size та Rent
4. Проаналізуйте взаємозв'язок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)


In [19]:
# Матриця кореляцій для числових змінних
# Створюємо датафрейм тільки з числовими метриками
metrics_df = df_clean[['BHK', 'Rent', 'Size', 'Bathroom']]

# Матриця кореляцій
correlation_matrix = metrics_df.corr()

# Візуалізація кореляцій
fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale= 'Portland',
    title='Кореляція між метриками',
    labels=dict(color="Кореляція")
)
fig.update_layout(height=500)
fig.show()

Оренда має помірно-позитивний зв'язок зі всіма трьома числовими змінними, найбільше з Bathroom (0.51).

А з ВНК та Size майже однакову (0.40 та 0.39 відповідно)

In [20]:
px.scatter(df_clean, x='Size', y='Rent', opacity=0.5,
           title='Площа vs Вартість оренди').show()

На scatter plot видно помірно-позитивний зв'язок, особливо до квартир із Size до 1250 кв футів. Але і розкид даних в цій частині графіка великий.

In [21]:
px.box(df_clean, x='BHK', y='Rent', title='Вартість оренди за кількістю BHK').show()

Медіанна вартість оренди зростає зі збільшенням кількості кімнат (BHK).

Найбільший розкид цін у 1–3 кімнатних квартир.

Із цікавого, максимальне значення оренди майже однакове для всіх ВНК (від 60 тис до 67 тис), крім ВНК = 5 (50 тис).

## Завдання 4: Feature Engineering та підготовка даних (4 бали)

**Що потрібно зробити:**
1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding. Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками. Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).
2. **Опціонально (по 0.5 бала за кожну доцільну ознаку):** Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі
3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.
4. Розділіть дані на ознаки (X) та цільову змінну (y)
5. Застосуйте стандартизацію до числових ознак


In [23]:
# One-Hot Encoding
cat_cols = ['Area Type', 'City', 'Furnishing Status', 'Tenant Preferred', 'Point of Contact']
dummies = pd.get_dummies(df_clean[cat_cols], drop_first=True, dtype=int)


In [24]:
# Додаємо до основного датафрейму
df_clean = pd.concat([df_clean, dummies], axis=1)

In [25]:
df_clean.head()

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,...,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Point of Contact_Contact Builder,Point of Contact_Contact Owner
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,...,0,0,1,0,0,1,1,0,0,1
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,...,0,0,1,0,1,0,1,0,0,1
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,...,0,0,1,0,1,0,1,0,0,1
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,...,0,0,1,0,0,1,1,0,0,1
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,...,0,0,1,0,0,1,0,0,0,1


In [26]:
# 3. Вибір ознак: виключаємо колонки з забагато значень, текстові та цільову
drop_cols = ['Rent', 'Posted On', 'Floor', 'Area Locality'] + cat_cols
features = df_clean.drop(columns=drop_cols)
features.dtypes   # перевірка: всі колонки мають бути числові

,0
BHK,int64
Size,int64
Bathroom,int64
Area Type_Carpet Area,int64
Area Type_Super Area,int64
City_Chennai,int64
City_Delhi,int64
City_Hyderabad,int64
City_Kolkata,int64
City_Mumbai,int64


In [27]:
# Розділення даних на ознаки (X) та цільову змінну (y)
X = features # Ознаки
y = df_clean['Rent']  # Цільова змінна

print(f"Розмір X (ознаки): {X.shape}")
print(f"Розмір y (цілі): {y.shape}")

Розмір X (ознаки): (4068, 16)
Розмір y (цілі): (4068,)


In [29]:
# 5. Стандартизація тільки числових (не dummy) ознак
from sklearn.preprocessing import StandardScaler

num_cols = ['BHK', 'Size', 'Bathroom']
scaler = StandardScaler()
X_scaled_df = X.copy()
X_scaled_df[num_cols] = scaler.fit_transform(X[num_cols])
X_scaled_df.head()

,BHK,Size,Bathroom,Area Type_Carpet Area,Area Type_Super Area,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Point of Contact_Contact Builder,Point of Contact_Contact Owner
0,0.048314,0.422549,0.259263,0,1,0,0,0,1,0,0,1,1,0,0,1
1,0.048314,-0.217329,-1.141376,0,1,0,0,0,1,0,1,0,1,0,0,1
2,0.048314,0.209256,-1.141376,0,1,0,0,0,1,0,1,0,1,0,0,1
3,0.048314,-0.217329,-1.141376,0,1,0,0,0,1,0,0,1,1,0,0,1
4,0.048314,-0.110683,-1.141376,1,0,0,0,0,1,0,0,1,0,0,0,1


## Завдання 5: Розділення даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на навчальну (80%) та тестову (20%) вибірки.
2. Створіть модель лінійної регресії.
3. Навчіть модель на навчальних даних.
4. Виведіть усі коефіцієнти моделі (ваги) та напишіть, які 2 ознаки найбільше впливають на прогноз.
5. Зробіть прогнози на тренувальній та тестовій вибірках.

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [31]:
# 1.Розділ даних
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y,
    test_size=0.2,  # 20% даних йде на тест
    random_state=42  # фіксуємо випадковість для відтворюваності
)

In [32]:
# 2.Створення моделі
model = LinearRegression()

In [33]:
# 3.Навчання моделі
model.fit(X_train, y_train)

LinearRegression()

In [34]:
# 4.Вивід ваг для кожної ознаки
for feature, weight in zip(model.feature_names_in_, model.coef_):
    print(f"{feature}: {weight:.2f}")

print(f"\nЗміщення (intercept): {model.intercept_:.2f}")

BHK: 2474.57
Size: 4138.51
Bathroom: 1743.05
Area Type_Carpet Area: 1066.94
Area Type_Super Area: 734.04
City_Chennai: -1135.20
City_Delhi: 1943.03
City_Hyderabad: -2509.85
City_Kolkata: -3481.41
City_Mumbai: 18385.94
Furnishing Status_Semi-Furnished: -4002.23
Furnishing Status_Unfurnished: -5181.32
Tenant Preferred_Bachelors/Family: -914.23
Tenant Preferred_Family: -2556.21
Point of Contact_Contact Builder: -5378.95
Point of Contact_Contact Owner: -8887.35

Зміщення (intercept): 28359.04


Найбільше на прогноз впливають:

1. City_Mumbai (+18 386): за інших рівних умов оренда в Mumbai приблизно на 18.4 тис. дорожча, ніж у Bangalore (базова категорія, видалена через drop_first).

2. Point of Contact_Contact Owner (−8 887): оголошення від власника приблизно на 8.9 тис. дешевші, ніж від агента (базова категорія).

In [35]:
# Прогнози на тренувальній та тестовій вибірках
# Прогнози на навчальній вибірці
y_train_pred = model.predict(X_train)

# Прогнози на тестовій вибірці
y_test_pred = model.predict(X_test)


# Порівняння перших 10 прогнозів з реальністю
comparison = pd.DataFrame({
    'Реальна ціна оренди': y_test.values[:10],
    'Прогнозовані ціна оренди': y_test_pred[:10].round(0),
    'Помилка': (y_test.values[:10] - y_test_pred[:10]).round(0),
    'Помилка %': ((y_test.values[:10] - y_test_pred[:10])/y_test.values[:10]*100).round(0)
})
print("Приклади прогнозів на тестовій вибірці:")
print(comparison)


Приклади прогнозів на тестовій вибірці:
   Реальна ціна оренди  Прогнозовані ціна оренди  Помилка  Помилка %
0                13000                   19887.0  -6887.0      -53.0
1                17000                   14217.0   2783.0       16.0
2                16500                   21170.0  -4670.0      -28.0
3                22000                   34527.0 -12527.0      -57.0
4                15000                   18741.0  -3741.0      -25.0
5                 4500                    6444.0  -1944.0      -43.0
6                40000                   23773.0  16227.0       41.0
7                55000                   34214.0  20786.0       38.0
8                 7000                    7810.0   -810.0      -12.0
9                 8000                    5964.0   2036.0       25.0


Помилки від 12% до 57%; модель недооцінює дорогі квартири (40–55 тис.) на 16–21 тис.

## Завдання 6: Оцінка якості моделі (2 бали)

**Що потрібно зробити:**
1. Обчисліть MAE, RMSE та R² для навчальної та тестової вибірок
2. Порівняйте метрики та зробіть висновок про якість моделі
3. Проаналізуйте і дайте висновок, чи є ознаки перенавчання або недонавчання (**Нагадування**: перенавчання - коли модель дуже добре працює на тренувальних даних, але погано на тестових; недонавчання - коли модель погано працює навіть на тренувальних даних)
4. Побудуйте графік розсіювання "реальні vs прогнозовані значення" та зробіть висновок про якість моделі


In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [37]:
def get_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R²': r2_score(y_true, y_pred),
    }

metrics = pd.DataFrame({
    'train': get_metrics(y_train, y_train_pred),
    'test':  get_metrics(y_test, y_test_pred),
}).T

metrics.round({'MAE': 0, 'RMSE': 0, 'R²': 3})


,MAE,RMSE,R²
train,5561.0,7718.0,0.697
test,5583.0,7721.0,0.683


In [38]:
np.mean(y_test)

np.float64(19343.528255528254)

 # Порівняння метрик

 MAE на тесті ≈ 5 583 рупій, тобто модель в середньому помиляється приблизно на 29% від середньої ціни оренди (19 344). Це помірна точність.

 RMSE (7 721) помітно більший за MAE (~1.4 рази), отже серед прогнозів є окремі великі помилки.

 R² = 0.68: модель пояснює близько 68% варіації ціни оренди, що непогано для простої лінійної моделі.

# Перенавчання / недонавчання

Метрики на тренувальній і тестовій вибірках майже однакові (R² 0.697 vs 0.683), тож перенавчання немає. Проте навіть на тренувальних даних модель пояснює лише ~70% варіації.
Це ознаки легкого недонавчання: лінійна модель з поточними ознаками занадто проста, щоб повністю описати залежність ціни.

In [39]:
# Візуалізація: реальні vs прогнозовані значення
import plotly.graph_objects as go

fig = px.scatter(
    x=y_test,
    y=y_test_pred,
    title='Реальні vs Прогнозовані ціни оренди',
    labels={'x': 'Реальні ціни оренди', 'y': 'Прогнозовані ціни оренди'},
    opacity=0.8
)

# Додаємо ідеальну лінію, де прогноз = реальність
max_val = max(y_test.max(), y_test_pred.max())
fig.add_trace(
  go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        name='Ідеальний прогноз',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(height=450)
fig.show()

Розкид прогнозованої ціни дещо збільшується для більш дорогих квартир (більше 30 тис рупій): модель їх починає недооцінювати із зростанням вартості оренди.

Систематичне відхилення вниз від лінії для дорогих квартир підтверджує недонавчання моделі.



## Завдання 7: Аналіз помилок (4 бали)

**Що потрібно зробити:**
1. Обчисліть помилки (residuals = реальні - прогнозовані значення)
2. Побудуйте гістограму розподілу помилок
3. Створіть scatter plot помилок відносно величини прогнозованих значень. Чи росте помилка з ростом прогнозованого значення?
4. Знайдіть 5 прогнозів з найбільшими помилками
5. Проаналізуйте, на яких типах житла модель помиляється найбільше. Типи можна розрізняти за кількістю кімнат чи містом, наприклад.
6. Подумайте і напишіть, які наступні кроки ви б зробили, аби поліпшити якість моделі. Опціонально можна їх зробити і ми перевіримо :)

In [40]:
# 1.Розраховуємо помилки (залишки)
residuals_test = y_test - y_test_pred

# 2.Гістограма помилок
fig = px.histogram(
    x=residuals_test,
    nbins=50,
    title='Розподіл помилок прогнозування: тестові дані',
    labels={'x': 'Помилка (реальні - прогнозовані)', 'count': 'Кількість'},
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Ідеальний прогноз")
fig.update_layout(height=450)
fig.show()

Висновок:

Помилки зосереджені навколо нуля, більшість лежить у межах ±10 тис., тобто в середньому модель не зміщена. Переоцінок трохи більше, ніж недооцінок, але вони здебільшого невеликі.

Розподіл скошений вправо: правий хвіст (+10…+30 тис.) довший і щільніший за лівий. Отже, великі помилки — це переважно недооцінка дорогих квартир.

In [41]:
# 3.Scatter plot: помилки vs прогнозовані значення
fig = px.scatter(
    x=y_test_pred,
    y=residuals_test,
    title='Залежність помилок від прогнозованих значень',
    labels={'x': 'Прогнозовані ціни оренди', 'y': 'Помилка'},
    opacity=0.8
)

# Додаємо горизонтальну лінію на 0
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Без помилки")

fig.update_layout(height=450)
fig.show()

Висновок:

Так, помилка зростає з ростом прогнозованого значення.

Для прогнозів до ~10 тис. помилки здебільшого в межах ±10 тис., а для прогнозів 30–45 тис. розкид сягає ±30 тис. (графік має форму «воронки»).

Тобто чим дорожчу квартиру прогнозує модель, тим менш точний прогноз в абсолютних значеннях.

In [42]:
# 4.Оренди з найбільшими помилками
# база помилок в тестовий даних
errors_df = pd.DataFrame({
    'real': y_test.values,
    'predicted': y_test_pred,
    'error': np.abs(residuals_test)
})

# Топ-5 найбільших помилок
top_errors = errors_df.nlargest(5, 'error')
print("Ціна оренди з найбільшими помилками прогнозування:")
print(top_errors.round(2))

Ціна оренди з найбільшими помилками прогнозування:
       real  predicted     error
3912  60000   26145.80  33854.20
4696  30000   63022.01  33022.01
275   30000   61674.79  31674.79
1090  65000   34538.91  30461.09
840   65000   34538.91  30461.09


Модель найбільше помилилась в абсолютних значеннях (рупіях) на:
1) дорогих квартирах ≥ 60 000 рупій (індекси 3912, 1090, 840)- які вона недооцінила;
2) 2 квартири вартістю 30 000 рупій  (індекси 4696, 275) - переоцінила.
Спробуємо проаналізувати у % значеннях.

In [43]:
errors_df['% of error'] = errors_df['error'] / errors_df['real']*100

top_rel_errors = errors_df.nlargest(5, '% of error')
print(top_rel_errors.round(2))

      real  predicted     error  % of error
944   5500   30333.26  24833.26      451.51
146   2200   10833.96   8633.96      392.45
1150  8000   38184.03  30184.03      377.30
2954  5000   15961.10  10961.10      219.22
267   6000   18649.14  12649.14      210.82


З даних виглядає так, що модель дуже сильно переоцінює оренду дешевих квартир.

Подивимося на них  в початковому наборі даних (див. внизу):

In [44]:
df.loc[[944, 146, 1150, 2954, 267]]

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
944,2022-06-20,1,5500,385,3 out of 9,Carpet Area,"Manibhadra Avenue, Morya Nagar",Mumbai,Unfurnished,Family,1,Contact Agent
146,2022-06-05,2,2200,550,2 out of 3,Super Area,Behala silpara,Kolkata,Furnished,Bachelors/Family,1,Contact Owner
1150,2022-06-20,2,8000,623,6 out of 7,Carpet Area,"Shreenath nagar, Morya Nagar",Mumbai,Unfurnished,Family,2,Contact Agent
2954,2022-07-04,1,5000,300,Ground out of 4,Carpet Area,Laxmi Nagar,Delhi,Semi-Furnished,Bachelors/Family,1,Contact Agent
267,2022-06-24,1,6000,2160,1 out of 2,Super Area,Kasba,Kolkata,Semi-Furnished,Bachelors,1,Contact Owner


Прогнозована ціна квартири за індексом 267 сильно відрізняється від тої, що в оголошенні, бо саме оголошення виглядає аномальним (2160 кв футів за 6000 рупій - низька ціна за таку велику квартиру; скоріше за все тут помилка вводу, про яку модель не знала)

Індекси 944 та 1150 знаходяться в дешевому районі Morya Nagar в Мумбаї. Модель знала лише про те, що Мумбаї дороге місто без врахування району. Тут є пункт для покращення моделі.

In [45]:
# Аналіз помилок за типами квартир
analysis = df_clean.loc[y_test.index, ['City', 'BHK', 'Furnishing Status']].copy()
analysis['real'] = y_test
analysis['abs_error'] = residuals_test.abs()
analysis['pct_error'] = analysis['abs_error'] / analysis['real'] * 100

# По містах
print(f'Аналіз помилок по містах\n')
display(analysis.groupby('City')[['abs_error', 'pct_error']].agg(['mean', 'count']).round(0))

Аналіз помилок по містах



abs_error       pct_error      
               mean count      mean count
City                                     
Bangalore    4331.0   171      29.0   171
Chennai      4642.0   162      31.0   162
Delhi        6868.0    89      42.0    89
Hyderabad    5605.0   143      37.0   143
Kolkata      4317.0   128      47.0   128
Mumbai       8979.0   121      35.0   121

Найбільше модель помиляється в ціні квартир у таких містах:
- Мумбаї: середня абсолютна помилка найбільша серед міст (~9 тис)
- Калькутта: середня відносна помилка найбільша серед міст (47%)

Різниця пояснюється рівнем цін: у Мумбаї оренда найвища, тому помилки в рупіях більші; у Калькутті найнижча, тому навіть невелика помилка дає великий відсоток.

Делі на другому місці за обома показниками (6.9 тис., 42%).

In [46]:
# По кількості кімнат
print(f'\nАналіз помилок по кількості кімнат\n')
display(analysis.groupby('BHK')[['abs_error', 'pct_error']].agg(['mean', 'count']).round(0))


Аналіз помилок по кількості кімнат



abs_error       pct_error      
         mean count      mean count
BHK                                
1      5274.0   231      45.0   231
2      4748.0   416      32.0   416
3      7717.0   153      30.0   153
4     11395.0    12      42.0    12
6     16736.0     2      55.0     2

 Найбільші середні помилки мають 6- та 4-кімнатні квартири (~16.7 і ~11.4 тис.), але їх у тестовій вибірці лише 2 і 12, тому ці значення ненадійні: одна велика помилка суттєво змінює середнє.

 Серед груп з достатньою кількістю даних найбільша відносна помилка в 1-кімнатних квартир (45% на 231 квартирі) проти 30–32% для 2- і 3-кімнатних. Ціни на 1-кімнатні низькі, тому навіть помилка в кілька тисяч дає великий відсоток.

Підсумок:

В абсолютних значеннях модель найбільше помиляється на дорогому житлі (Mumbai, великі квартири), у відносних — на дешевому (Kolkata, 1-кімнатні квартири).

# 6. Наступні кроки для покращення моделі:
1. Логарифмувати цільову змінну (log(Rent)): ціни мають скошений розподіл, а помилка зростає з ціною. Логарифм зменшить вплив дорогих квартир і їх недооцінку.
2. Додати інформацію про район (Area Locality): модель не розрізняє дешеві та дорогі райони одного міста. Можна застосувати target encoding, обчислюючи середні лише на навчальній вибірці, щоб уникнути витоку даних
3. Розпарсити Floor: поверх і загальна кількість поверхів (висотні будинки зазвичай дорожчі).
4. Взаємодії ознак: Size × City, бо вплив площі на ціну різний у різних містах (метр у Mumbai дорожчий, ніж у Kolkata).
5. Краще очистити аномалії: записи на кшталт 1 BHK / 2160 sq ft / 6000 рупій.




